In [ ]:
import io
import os
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as et
import requests

from databricks import sql
from dotenv import load_dotenv

In [14]:
# Carrega as variáveis de ambiente do .env
load_dotenv() 

server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME")
http_path = os.getenv("DATABRICKS_HTTP_PATH")
access_token = os.getenv("DATABRICKS_TOKEN")

In [ ]:
host_limpo = server_hostname.replace("https://", "").rstrip("/")
caminho_local_parquet = "divvy_tripdata_consolidado.parquet"
caminho_volume = "/Volumes/cyclistic/bronze/raw_files/divvy_tripdata_consolidado.parquet"

url = f"https://{host_limpo}/api/2.0/fs/files{caminho_volume}"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/octet-stream"
}

print(f"Fazendo stream do arquivo para {url}...")

with open(caminho_local_parquet, "rb") as f:
    response = requests.put(url, headers=headers, data=f)

if response.status_code in (200, 204):
    print("✅ Upload concluído com sucesso via API!")
else:
    print(f"❌ Erro {response.status_code}: {response.text}")

Fazendo stream do arquivo para https://dbc-95c29010-daf3.cloud.databricks.com/api/2.0/fs/files/Volumes/cyclistic/bronze/raw_files/divvy_tripdata_consolidado.parquet...
✅ Upload concluído com sucesso via API!


In [ ]:
caminho_volume = "/Volumes/cyclistic/bronze/raw_files/divvy_tripdata_consolidado.parquet"

with sql.connect(
    server_hostname=server_hostname,
    http_path=http_path,
    access_token=access_token
) as connection:
    with connection.cursor() as cursor:
        print("Criando tabela Delta 'cyclistic.bronze.tripdata_raw'...")
        
        # Cria ou substitui a tabela lendo direto do arquivo Parquet no Volume
        cursor.execute(f"""
            CREATE OR REPLACE TABLE cyclistic.bronze.tripdata_raw AS
            SELECT * FROM read_files(
                '{caminho_volume}',
                format => 'parquet'
            );
        """)
        
        # Validação da contagem de linhas
        cursor.execute("SELECT COUNT(*) AS total_linhas FROM cyclistic.bronze.tripdata_raw;")
        resultado = cursor.fetchall()
        total = resultado[0]['total_linhas']
        
        print(f"🎉 Camada Bronze finalizada com sucesso!")
        print(f"📊 Total de registros ingeridos: {total:,}".replace(",", "."))

Criando tabela Delta 'cyclistic.bronze.tripdata_raw'...
🎉 Camada Bronze finalizada com sucesso!
📊 Total de registros ingeridos: 34.900.846
